<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Portfolio-Management/blob/main/Cost_of_Trading_%26_Market_Impact_(Chapter_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The Square-Root Law of Market Impact

Linear costs (like exchange fees) are easy to model. Market Impact is nonlinear and much more dangerous. Isichenko outlines that the price impact of a trade roughly follows a square-root relationship with volume.

The industry-standard approximation for temporary market impact is:

$$\text{Impact} = c \cdot \sigma \cdot \sqrt{\frac{Q}{V}}$$

* $\sigma$: The daily volatility of the stock.
* $Q$: The size of your order (in dollars or shares).
* $V$: The Average Daily Volume (ADV) of the stock.
* $c$: A universal constant (often estimated around 0.1 for liquid US equities).

### B. The Institutional Reality

If your XGBoost model predicts a 15 basis point (bp) edge on a stock, but you try to push $50 million into a stock that only trades
\\$100 million a day, your own buying pressure will drive the price up by 30 bps before you finish executing. You just turned a winning signal into a guaranteed loss.

## The Slippage Simulator

In [40]:
import xgboost as xgb
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [41]:

universe = {
    'AAPL': 'Tech', 'MSFT': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech', 'ORCL': 'Tech',
    'JPM': 'Fin', 'BAC': 'Fin', 'GS': 'Fin', 'MS': 'Fin', 'C': 'Fin',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'EOG': 'Energy', 'SLB': 'Energy',
    'JNJ': 'Health', 'UNH': 'Health', 'PFE': 'Health', 'ABBV': 'Health', 'MRK': 'Health'
}
tickers = list(universe.keys())

print("Fetching historical data... (Takes ~10 seconds)")
# Fetch 1 year of daily close prices
prices = yf.download(tickers, period="1y")['Close']
prices = prices[tickers] # Ensure column order

[                       0%                       ]

Fetching historical data... (Takes ~10 seconds)


[*********************100%***********************]  20 of 20 completed


In [42]:
# We use .shift(-1) because today's features must predict tomorrow's return
raw_returns = prices.pct_change()
forward_returns = raw_returns.shift(-1)

In [43]:
# Convert to a "Long" format DataFrame (Standard for ML pipelines)
df = forward_returns.unstack().reset_index()
df.columns = ['Ticker', 'Date', 'Fwd_Return']
df = df.dropna()


In [44]:
# Map the sectors
df['Sector'] = df['Ticker'].map(universe)

In [45]:
def neutralize_and_rank(daily_data):
    # Market Neutralization (Subtract cross-sectional mean)
    daily_data['Market_Mean'] = daily_data['Fwd_Return'].mean()
    daily_data['Market_Neutral'] = daily_data['Fwd_Return'] - daily_data['Market_Mean']

    # Sector Neutralization (Subtract sector mean from the market-neutral returns)
    sector_means = daily_data.groupby('Sector')['Market_Neutral'].transform('mean')
    daily_data['Idiosyncratic_Return'] = daily_data['Market_Neutral'] - sector_means

    # Rank Normalization (Scale between 0 and 1 to suppress outliers)
    daily_data['ML_Target'] = daily_data['Idiosyncratic_Return'].rank(pct=True)

    return daily_data

print("Applying Cross-Sectional Neutralization and Rank Normalization...")
# Apply the function group-by-group for every single day in the dataset
ml_dataset = df.groupby('Date', group_keys=False).apply(neutralize_and_rank)

Applying Cross-Sectional Neutralization and Rank Normalization...


In [46]:
# We must sort by Ticker and Date to calculate rolling features correctly
ml_dataset = ml_dataset.sort_values(['Ticker', 'Date'])

In [47]:
# Short-Term Mean Reversion (5-Day Return)
# Hypothesis: High 5-day return means it's overbought and will revert (negative weight expected)
ml_dataset['F_Reversion_5d'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(5).sum())

# Medium-Term Momentum (21-Day Return)
# Hypothesis: High 1-month return means it's trending (positive weight expected)
ml_dataset['F_Momentum_1m'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).sum())

# Daily Volatility (21-Day standard deviation)
ml_dataset['F_Volatility'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).std())

ml_dataset = ml_dataset.dropna()

print("Cross-Sectional Feature Standardization (Z-Scoring)...")
# Just like targets, features must be standardized cross-sectionally every single day
features = ['F_Reversion_5d', 'F_Momentum_1m', 'F_Volatility']

Cross-Sectional Feature Standardization (Z-Scoring)...


In [48]:
def standardize_features(daily_data):
    for f in features:
        daily_data[f] = (daily_data[f] - daily_data[f].mean()) / (daily_data[f].std() + 1e-8)
    return daily_data

ml_dataset = ml_dataset.groupby('Date', group_keys=False).apply(standardize_features)

In [49]:

print("Initializing the XGBoost Regressor...")
# MFE Constraints for Financial Data:
# 1. max_depth=3 (Prevent memorizing noise)
# 2. learning_rate=0.05 (Learn slowly)
# 3. subsample=0.8 (Train on 80% of data per tree to prevent overfitting)
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

Initializing the XGBoost Regressor...


In [50]:
# X is our features, y is our Neutralized Rank Target from Day 1
X = ml_dataset[features]
y = ml_dataset['ML_Target']

In [51]:
print("Training the Non-Linear Ensemble...")
xgb_model.fit(X, y)

Training the Non-Linear Ensemble...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [54]:
tickers = list(ml_dataset['Ticker'].unique())
raw_data = yf.download(tickers, period="3mo")

# 1. Calculate Raw 21-day Average Daily Dollar Volume (ADV)
close_prices = raw_data['Close']
volumes = raw_data['Volume']
dollar_volumes = close_prices * volumes
adv_21d = dollar_volumes.rolling(21).mean().iloc[-1] # Latest ADV

# 2. Calculate RAW 21-day Volatility (The Fix)
# We calculate the actual standard deviation of daily returns
recent_returns = close_prices.pct_change()
raw_volatility_21d = recent_returns.rolling(21).std().iloc[-1]
volatility = raw_volatility_21d # Assign the raw volatility to the expected variable name

# Get the theoretical XGBoost Alpha (Prediction)
ml_dataset['XGB_Prediction'] = xgb_model.predict(X)
latest_date = ml_dataset['Date'].max()
latest_data = ml_dataset[ml_dataset['Date'] == latest_date].set_index('Ticker')
xgb_alpha = latest_data['XGB_Prediction']

[*********************100%***********************]  20 of 20 completed


In [55]:
print("Simulating Institutional Execution...")
# Define Portfolio Parameters
PORTFOLIO_AUM = 100_000_000  # $100 Million
TARGET_WEIGHT = 0.05         # 5% target allocation per stock
ORDER_SIZE = PORTFOLIO_AUM * TARGET_WEIGHT # $5M order per stock

# The Square-Root Impact Model
c_constant = 0.1
impact_costs = []

print("\nMARKET IMPACT ANALYSIS (Target Order: $5M per stock):")
print("="*75)
print(f"{'Ticker':<10} | {'ADV ($M)':<12} | {'ML Alpha (bps)':<15} | {'Slippage (bps)':<15} | {'Net Alpha':<10}")
print("-" * 75)

for ticker in tickers:
    if ticker not in adv_21d or ticker not in volatility:
        continue

    stock_adv = adv_21d[ticker]
    stock_vol = volatility[ticker]
    theo_alpha = xgb_alpha[ticker] * 10000 # Convert to basis points (bps)

    # Impact = c * Volatility * sqrt(Order Size / ADV)
    # Note: We use raw decimal volatility for the formula, then convert output to bps
    impact_decimal = c_constant * stock_vol * np.sqrt(ORDER_SIZE / stock_adv)
    impact_bps = impact_decimal * 10000

    net_alpha = theo_alpha - impact_bps

    # Format for display
    adv_m = stock_adv / 1_000_000
    print(f"{ticker:<10} | ${adv_m:>10.1f}M | {theo_alpha:>12.2f} bps | {-impact_bps:>13.2f} bps | {net_alpha:>9.2f} bps")

print("="*75)

Simulating Institutional Execution...

MARKET IMPACT ANALYSIS (Target Order: $5M per stock):
Ticker     | ADV ($M)     | ML Alpha (bps)  | Slippage (bps)  | Net Alpha 
---------------------------------------------------------------------------
AAPL       | $   14834.7M |      5306.62 bps |         -0.28 bps |   5306.33 bps
ABBV       | $    1131.2M |      5058.00 bps |         -0.99 bps |   5057.01 bps
AMD        | $   15406.1M |      5667.60 bps |         -0.94 bps |   5666.66 bps
BAC        | $    1997.1M |      5110.08 bps |         -0.58 bps |   5109.50 bps
C          | $    1469.5M |      5267.94 bps |         -1.04 bps |   5266.90 bps
COP        | $     781.6M |      5303.78 bps |         -1.49 bps |   5302.29 bps
CVX        | $    1669.5M |      5319.07 bps |         -0.89 bps |   5318.17 bps
EOG        | $     454.2M |      5284.80 bps |         -1.93 bps |   5282.87 bps
GS         | $    2372.1M |      5215.19 bps |         -1.16 bps |   5214.02 bps
JNJ        | $    1695.6M |